# Sleep Architecture: R to AASM Stage Mapping

The `sleep_architecture` domainpack models cortical EEG bands
(delta, theta, alpha, gamma) as coupled Kuramoto oscillators.
The global order parameter R maps to AASM sleep stages
(Iber et al. 2007):

| Stage | R range  | Cortical state                              |
|-------|----------|---------------------------------------------|
| N3    | >= 0.70  | Slow-wave sleep, highly synchronised delta   |
| N2    | 0.40-0.70| Moderate sync, spindles and K-complexes       |
| N1    | 0.30-0.40| Light sleep, partial desynchronisation        |
| REM   | 0.20-0.35| Low R + functional desynchronisation flag     |
| Wake  | < 0.30   | Desynchronised cortex (without functional_desync) |

The **ultradian cycle** (~90 minutes) governs NREM-REM alternation.
We estimate cycle phase from the most recent N3 epoch
(Rechtschaffen & Kales 1968).

In [ ]:
import numpy as np

from scpn_phase_orchestrator.coupling.knm import CouplingBuilder
from scpn_phase_orchestrator.monitor.sleep_staging import (
    classify_sleep_stage,
    ultradian_phase,
)
from scpn_phase_orchestrator.upde.engine import UPDEEngine
from scpn_phase_orchestrator.upde.order_params import compute_order_parameter

In [ ]:
# Set up the sleep_architecture domainpack.
# 8 oscillators: 2 delta (0.5 Hz), 2 theta (4 Hz), 2 alpha (8 Hz), 2 gamma (40 Hz)
N_OSC = 8
DT = 0.01
STEPS = 1000
TWO_PI = 2.0 * np.pi

# Natural frequencies from binding_spec (rad/s)
omegas = np.array(
    [
        3.14,
        3.14,  # delta
        25.13,
        25.13,  # theta
        50.27,
        50.27,  # alpha
        251.33,
        251.33,  # gamma
    ]
)

coupling = CouplingBuilder().build(
    n_layers=N_OSC,
    base_strength=0.4,  # from binding_spec coupling.base_strength
    decay_alpha=0.25,  # from binding_spec coupling.decay_alpha
)
knm = coupling.knm
alpha = coupling.alpha

# Start with random phases (simulating wake state)
rng = np.random.default_rng(42)
phases = rng.uniform(0, TWO_PI, N_OSC)

engine = UPDEEngine(N_OSC, dt=DT)

# Simulate with varying coupling to drive sleep stage transitions.
# Increase coupling over time to simulate sleep onset (Wake -> N1 -> N2 -> N3),
# then decrease for REM, then increase again.
R_hist = []
stage_hist = []
timestamps = []

for step in range(STEPS):
    t = step * DT
    timestamps.append(t)

    # Modulate coupling to drive sleep cycles
    # 0-200: wake -> sleep onset (ramp up K)
    # 200-500: deep sleep (high K)
    # 500-700: lightening toward REM (ramp down K)
    # 700-850: REM (low K)
    # 850-1000: back toward deep sleep
    if step < 200:
        k_scale = 0.5 + 2.5 * (step / 200)
    elif step < 500:
        k_scale = 3.0
    elif step < 700:
        k_scale = 3.0 - 2.2 * ((step - 500) / 200)
    elif step < 850:
        k_scale = 0.8
    else:
        k_scale = 0.8 + 2.2 * ((step - 850) / 150)

    scaled_knm = knm * k_scale
    phases = engine.step(phases, omegas, scaled_knm, 0.0, 0.0, alpha)

    R, _ = compute_order_parameter(phases)
    R_hist.append(R)

    # REM flag: low coupling phase with gamma activity
    functional_desync = 700 <= step < 850
    stage = classify_sleep_stage(R, functional_desync=functional_desync)
    stage_hist.append(stage)

print(f"Simulation: {STEPS} steps, {STEPS * DT:.1f}s")
print(f"Final R: {R_hist[-1]:.4f}, stage: {stage_hist[-1]}")

In [ ]:
# Classify sleep stage at each step from R values
stage_counts = {}
for s in stage_hist:
    stage_counts[s] = stage_counts.get(s, 0) + 1

print("Sleep stage distribution:")
for stage in ["Wake", "N1", "N2", "N3", "REM"]:
    count = stage_counts.get(stage, 0)
    pct = 100 * count / STEPS
    print(f"  {stage:4s}: {count:4d} epochs ({pct:5.1f}%)")

# Show transitions
transitions = 0
for i in range(1, len(stage_hist)):
    if stage_hist[i] != stage_hist[i - 1]:
        transitions += 1
print(f"\nTotal stage transitions: {transitions}")

In [ ]:
# Plot R trajectory with sleep stage annotations
try:
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    stage_colors = {
        "Wake": "#e74c3c",
        "N1": "#f39c12",
        "N2": "#3498db",
        "N3": "#2c3e50",
        "REM": "#27ae60",
    }

    t = np.array(timestamps)
    fig, axes = plt.subplots(
        2, 1, figsize=(12, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]}
    )

    # R trajectory
    axes[0].plot(t, R_hist, linewidth=0.8, color="black")
    # Threshold lines
    for thresh, label in [(0.70, "N3"), (0.40, "N2"), (0.30, "N1"), (0.20, "REM")]:
        axes[0].axhline(thresh, color="gray", linestyle=":", linewidth=0.5, alpha=0.7)
        axes[0].text(t[-1] + 0.1, thresh, label, fontsize=7, va="center")
    axes[0].set_ylabel("R (order parameter)")
    axes[0].set_ylim(-0.05, 1.1)
    axes[0].set_title("Sleep Architecture: R Trajectory with AASM Staging")

    # Stage coloring in hypnogram style
    stage_to_y = {"Wake": 4, "REM": 3, "N1": 2, "N2": 1, "N3": 0}
    y_vals = [stage_to_y[s] for s in stage_hist]
    for i in range(len(t) - 1):
        axes[1].fill_between(
            [t[i], t[i + 1]],
            y_vals[i],
            y_vals[i] + 0.8,
            color=stage_colors[stage_hist[i]],
            alpha=0.8,
        )
    axes[1].set_yticks([0.4, 1.4, 2.4, 3.4, 4.4])
    axes[1].set_yticklabels(["N3", "N2", "N1", "REM", "Wake"])
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Stage")
    axes[1].set_ylim(-0.2, 5)

    # Legend
    handles = [Patch(facecolor=c, label=s) for s, c in stage_colors.items()]
    axes[1].legend(handles=handles, loc="upper right", fontsize=7, ncol=5)

    fig.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed, skipping plot")

In [ ]:
# Compute ultradian phase
# The ultradian cycle (~90 min) tracks position within NREM-REM alternation.
# Phase 0 = N3 onset (deepest sleep), phase ~0.5 = mid-cycle (near REM).

ts_array = np.array(timestamps)
uph = ultradian_phase(ts_array, stage_hist)
print(f"Ultradian phase at end of simulation: {uph:.4f}")

# Track ultradian phase over time
uph_hist = []
for i in range(len(timestamps)):
    uph_i = ultradian_phase(ts_array[: i + 1], stage_hist[: i + 1])
    uph_hist.append(uph_i)

# Find N3 epochs
n3_onset_times = []
for i in range(1, len(stage_hist)):
    if stage_hist[i] == "N3" and stage_hist[i - 1] != "N3":
        n3_onset_times.append(timestamps[i])

print(f"N3 onsets at t = {[f'{t:.2f}' for t in n3_onset_times]}")

try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(ts_array, uph_hist, linewidth=0.8, color="tab:purple")
    for t_n3 in n3_onset_times:
        ax.axvline(t_n3, color="tab:blue", linestyle="--", linewidth=0.5, alpha=0.7)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Ultradian phase")
    ax.set_title("Ultradian Cycle Phase (0 = N3 onset, 0.5 = mid-cycle)")
    ax.set_ylim(-0.05, 1.05)
    fig.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed, skipping plot")